In [1]:
R.version.string

[1] "R version 4.5.1 (2025-06-13 ucrt)"

In [2]:
# # 1) Pick a user-writable library path (avoids OneDrive/Documents issues)
# v <- paste(R.version$major, strsplit(R.version$minor, "\\.")[[1]][1], sep = ".")
# userlib <- normalizePath(file.path(Sys.getenv("LOCALAPPDATA"),
#                                    "R", "win-library", v),
#                          winslash = "/")
# dir.create(userlib, recursive = TRUE, showWarnings = FALSE)

# # 2) Prepend to library search path for this session
# .libPaths(c(userlib, .libPaths()))

# # 3) Install
# install.packages("intergraph")
# # sanity check
# .libPaths()


In [3]:
library(statnet)
library(dplyr)
library(lubridate)

Loading required package: tergm

Loading required package: ergm

Loading required package: network


'network' 1.19.0 (2024-12-08), part of the Statnet Project
* 'news(package="network")' for changes since last version
* 'citation("network")' for citation information
* 'https://statnet.org' for help, support, and other information



'ergm' 4.10.1 (2025-08-26), part of the Statnet Project
* 'news(package="ergm")' for changes since last version
* 'citation("ergm")' for citation information
* 'https://statnet.org' for help, support, and other information


'ergm' 4 is a major update that introduces some backwards-incompatible
changes. Please type 'news(package="ergm")' for a list of major
changes.


Loading required package: networkDynamic


'networkDynamic' 0.11.5 (2024-11-21), part of the Statnet Project
* 'news(package="networkDynamic")' for changes since last version
* 'citation("networkDynamic")' for citation information
* 'https://statnet.org' for help, support, and other informati

In [4]:
nodes <- read.csv("nodes.csv", stringsAsFactors = FALSE)
edges <- read.csv("edges.csv", stringsAsFactors = FALSE)

In [5]:
node_ids <- nodes$mbid
g <- network.initialize(length(node_ids), directed = FALSE)
network.vertex.names(g) <- node_ids

In [6]:
class(g)

[1] "network"

In [7]:
## helpers for optional/std columns
pick_col <- function(df, ...) {
  for (nm in c(...)) if (nm %in% names(df)) return(df[[nm]])
  return(NULL)
}

# role/genre as character (NOT factor)
set.vertex.attribute(g, "name",         as.character(nodes$name))
set.vertex.attribute(g, "id", as.character(nodes$mbid))
set.vertex.attribute(g, "all_roles",    as.character(nodes$all_roles))
set.vertex.attribute(g, "role_major",   as.character(nodes$role_major))
set.vertex.attribute(g, "all_genres",   as.character(nodes$all_genres))
set.vertex.attribute(g, "primary_genre",as.character(nodes$primary_genre))

# dates as character (NOT Date/POSIX*)
set.vertex.attribute(g, "first_release_date_in_window",
                     as.character(nodes$first_release_date_in_window))
set.vertex.attribute(g, "last_release_date_in_window",
                     as.character(nodes$last_release_date_in_window))

# numeric columns (make sure they’re numeric vectors)
set.vertex.attribute(g, "num_songs_in_window",         as.numeric(nodes$num_songs_in_window))
set.vertex.attribute(g, "num_collaborators_in_window", as.numeric(nodes$num_collaborators_in_window))
set.vertex.attribute(g, "time_in_network_years",       as.numeric(nodes$time_in_network_years))

# standardized variants: pick available column or compute on the fly
ns_std <- pick_col(nodes, "num_songs_in_window_std", "num_songs_std")
if (is.null(ns_std)) ns_std <- scale(as.numeric(nodes$num_songs_in_window))
set.vertex.attribute(g, "num_songs_std", as.numeric(ns_std))

nc_std <- pick_col(nodes, "num_collaborators_in_window_std", "num_collab_std")
if (is.null(nc_std)) nc_std <- scale(as.numeric(nodes$num_collaborators_in_window))
set.vertex.attribute(g, "num_collab_std", as.numeric(nc_std))

t_std <- pick_col(nodes, "time_in_network_years_std", "time_std")
if (is.null(t_std)) t_std <- scale(as.numeric(nodes$time_in_network_years))
set.vertex.attribute(g, "time_std", as.numeric(t_std))

In [8]:
# helper for NULL-coalescing (if you used slightly different std column names)
`%||%` <- function(a, b) if (!is.null(a)) a else b

# ------------------------------------------------------------------
# 4) Add edges in bulk, preserving edge order
# ------------------------------------------------------------------
idx_u <- match(edges$u, node_ids)
idx_v <- match(edges$v, node_ids)

keep <- !is.na(idx_u) & !is.na(idx_v) & idx_u != idx_v
idx_u <- idx_u[keep]; idx_v <- idx_v[keep]
edges_kept <- edges[keep, , drop = FALSE]

add.edges(g, tail = idx_u, head = idx_v)

In [9]:
ok <- !is.na(idx_u) & !is.na(idx_v)

eids <- mapply(
  function(t, h) get.edgeIDs(g, v = t, alter = h, neighborhood = "combined"),
  idx_u[ok], idx_v[ok]
)

## If no parallel edges, each call should return a single ID; coerce to integer vector
eids <- as.integer(eids)

## Some pairs might have been skipped by edge.check=TRUE (duplicates).
## Keep only the successfully found edge IDs and align attributes accordingly.
keep <- !is.na(eids)
eids_kept <- eids[keep]


In [10]:
# eids_kept should be the vector of edge IDs that correspond to edges_kept rows
# e.g., from get.edgeIDs(...) or from the batch-add trick above

set.edge.attribute(g, "weight_raw",        as.numeric(edges_kept$weight_raw),        e = eids_kept)
set.edge.attribute(g, "weight_size_adj",   as.numeric(edges_kept$weight_size_adj),   e = eids_kept)
set.edge.attribute(g, "first_collab_date", as.character(edges_kept$first_collab_date), e = eids_kept)
set.edge.attribute(g, "last_collab_date",  as.character(edges_kept$last_collab_date),  e = eids_kept)
set.edge.attribute(g, "recency_weight",    as.numeric(edges_kept$recency_weight),    e = eids_kept)
set.edge.attribute(g, "roles_overlap",     as.numeric(edges_kept$roles_overlap),     e = eids_kept)
set.edge.attribute(g, "genre_overlap",     as.numeric(edges_kept$genre_overlap),     e = eids_kept)
if ("low_overlap" %in% names(edges_kept)) {
  set.edge.attribute(g, "low_overlap", as.numeric(edges_kept$low_overlap), e = eids_kept)
}


In [11]:
network.edgecount(g)
gden(g)                       # observed density
summary(g ~ edges + 
           gwesp(0.5, fixed=TRUE) + 
           gwdegree(1.5, fixed=TRUE))


[1] 7218

[1] 0.001834114

edges gwesp.fixed.0.5 gwdeg.fixed.1.5 
       7218.000       10663.596        7069.342

In [12]:
is.directed(g)                     # should be FALSE (you created undirected)
any(is.na(network.vertex.names(g)))
anyDuplicated(network.vertex.names(g))  # duplicated names can bite in some ops

# No self-loops and no missing endpoints
any(which(is.na(as.edgelist(g)))) 

[1] FALSE

[1] FALSE

[1] 0

[1] FALSE

control ergm. Note that g has:

vertex attributes: role_major, primary_genre, num_songs_in_window_std, time_in_network_years_std

edge attributes: low_overlap, roles_overlap, genre_overlap, etc.

In [13]:
# Robust core count
ncores    <- parallel::detectCores()
nworkers  <- if (is.na(ncores)) 1L else max(1L, ncores - 1L)

# Choose a parallel.type only if using >1 worker
ptype <- if (nworkers > 1L) {
  if (.Platform$OS.type == "windows") "PSOCK" else "FORK"
} else NULL

set.seed(123)

m0 <- ergm(g ~ edges + gwdegree(0.8, fixed=TRUE), estimate = "MPLE")

# Step B: add gwesp with a conservative decay and small (even negative) start
start <- c(
  edges   = coef(m0)["edges"],
  gwesp   = -0.5,                  # damp triangles to avoid blow-up at start
  gwdegree= coef(m0)["gwdegree.decay0.8"] %||% 0  # if absent, fallback 0
)

ctrl <- control.ergm(
  init.method         = "MPLE",
  MCMLE.density.guard = 100,       # bump a bit, but don’t rely on this
  MCMC.prop.weights   = "TNT",
  MCMC.burnin         = 5e4,
  MCMC.interval       = 1e3,
  MCMC.samplesize     = 2e3,
  parallel          = nworkers,
  parallel.type     = ptype,   # NULL if single-core
)


Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




In [14]:
ctrl_fast <- control.ergm(
  init.method       = "MPLE",
  MCMC.prop.weights = "TNT",
  # lighter sampler to get you moving
  MCMC.burnin       = 10000,
  MCMC.interval     = 200,
  MCMC.samplesize   = 1000,
  # cap how long MCMLE keeps iterating
  MCMLE.maxit       = 10,
  # parallel: avoid PSOCK overhead on Windows
  parallel          = if (.Platform$OS.type == "windows") 1L else nworkers,
  parallel.type     = if (.Platform$OS.type == "windows") NULL else "FORK"
)

simple ERGM, every dyad has the same probability of collaboration independent of anything else. 

H: is the network denser or sparser than pure randomness?

Single edges reflect baseline log odds of a tie, if edges = -4, each potential pair has exp(-4) = 0.018 probability of collaborating

In [15]:
theta0 <- c(qlogis(gden(g)), -0.5, -0.5)
sims <- simulate(g ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(1.5, fixed=TRUE),
                 coef = theta0, nsim = 20, output = "stats")
summary(sims[,"edges"])   # should be in the same ballpark as ~7

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   4741    4810    4898    4935    5010    5286 

geometrically weighted edgewise shared partners captures triadict closure (friends of friends tend to collaborate). A positive value will indicate strong clustering

gwdegress: geometrically weighted degree models skew of degree distribution (preferential attachment). A positive value will indicate that popular artists attract many collaborators

H: does collaboration cluster into triangles? do hubs form?

In [16]:
# m1 <- ergm(
#   g ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE),
#   control = ctrl
# )
# https://chatgpt.com/c/6913a752-f200-8327-85ec-e8949376ab0c
m1_fast <- ergm(g ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE), estimate="MPLE")

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




In [28]:
summary(m1_fast)

Call:
ergm(formula = g ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, 
    fixed = TRUE), estimate = "MPLE")

Maximum Pseudolikelihood Results:

                Estimate Std. Error MCMC % z value Pr(>|z|)    
edges           -8.29837    0.04361      0 -190.29   <1e-04 ***
gwesp.fixed.0.5  4.49811    0.02720      0  165.39   <1e-04 ***
gwdeg.fixed.0.8 -3.16005    0.08628      0  -36.63   <1e-04 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


     Null Pseudo-deviance: 5455644  on 3935415  degrees of freedom
 Residual Pseudo-deviance:   26578  on 3935412  degrees of freedom
 
AIC: 26584  BIC: 26624  (Smaller is better. MC Std. Err. = 0)

In [17]:
# https://chatgpt.com/c/6913a233-4de4-832f-93e7-ec75ff57b8e4
sims <- simulate(m1_fast, nsim = 2)


In [18]:
library(intergraph)
library(igraph)

Warning message:
"package 'intergraph' was built under R version 4.5.2"
Warning message:
"package 'igraph' was built under R version 4.5.2"

Attaching package: 'igraph'


The following objects are masked from 'package:lubridate':

    %--%, union


The following objects are masked from 'package:dplyr':

    as_data_frame, groups, union


The following objects are masked from 'package:sna':

    betweenness, bonpow, closeness, components, degree, dyad.census,
    evcent, hierarchy, is.connected, neighborhood, triad.census


The following objects are masked from 'package:network':

    %c%, %s%, add.edges, add.vertices, delete.edges, delete.vertices,
    get.edge.attribute, get.edges, get.vertex.attribute, is.bipartite,
    is.directed, list.edge.attributes, list.vertex.attributes,
    set.edge.attribute, set.vertex.attribute


The following objects are masked from 'package:stats':

    decompose, spectrum


The following object is masked from 'package:base':

    union




In [19]:

gi <- intergraph::asIgraph(g)     # convert from statnet network
gi <- simplify(gi, remove.loops = TRUE, remove.multiple = TRUE)

deg_obs <- igraph::degree(gi)     # undirected degree
tri_obs <- igraph::count_triangles(gi)  # per-vertex triangle counts

In [27]:
g

 Network attributes:
  vertices = 2806 
  directed = FALSE 
  hyper = FALSE 
  loops = FALSE 
  multiple = FALSE 
  bipartite = FALSE 
  total edges= 7218 
    missing edges= 0 
    non-missing edges= 7218 

 Vertex attribute names: 
    all_genres all_roles first_release_date_in_window id last_release_date_in_window name num_collab_std num_collaborators_in_window num_songs_in_window num_songs_std primary_genre role_major time_in_network_years time_std vertex.names 

 Edge attribute names not shown 

In [21]:

## 2) Simulations -> igraph, compute per-node degree/triangles, align by name
sims_list <- if (inherits(sims, "network")) list(sims) else sims

# helper to get a named vector from a simulated network
sim_metrics <- function(nw) {
  gj <- intergraph::asIgraph(nw)
  gj <- simplify(gj, remove.loops = TRUE, remove.multiple = TRUE)
  nm <- V(gj)$mbid
  if (is.null(nm)) nm <- as.character(seq_len(vcount(gj)))
  list(
    deg  = setNames(igraph::degree(gj), nm),
    tri  = setNames(igraph::count_triangles(gj), nm)
  )
}

In [22]:
# collect into matrices aligned to g_names
deg_mat <- matrix(NA_real_, nrow = length(g_names), ncol = length(sims_list),
                  dimnames = list(g_names, NULL))
tri_mat <- matrix(NA_real_, nrow = length(g_names), ncol = length(sims_list),
                  dimnames = list(g_names, NULL))

for (j in seq_along(sims_list)) {
  m <- sim_metrics(sims_list[[j]])
  deg_mat[g_names, j] <- m$deg[g_names]
  tri_mat[g_names, j] <- m$tri[g_names]
}

In [23]:
# Expected values (row means; NAs can appear only if names mismatched)
deg_exp <- rowMeans(deg_mat, na.rm = TRUE)
tri_exp <- rowMeans(tri_mat, na.rm = TRUE)


In [24]:

## 3) Assemble your data frame
node_summary <- data.frame(
  node = g_names,
  degree_obs = as.numeric(deg_obs[g_names]),
  degree_exp = as.numeric(deg_exp[g_names]),
  tri_obs    = as.numeric(tri_obs[g_names]),
  tri_exp    = as.numeric(tri_exp[g_names]),
  residual_degree = as.numeric(deg_obs[g_names] - deg_exp[g_names]),
  residual_tri    = as.numeric(tri_obs[g_names] - tri_exp[g_names]),
  row.names = NULL
)

In [25]:
g_names

NULL

In [26]:
head(node_summary)

degree_obs,degree_exp,tri_obs,tri_exp,residual_degree,residual_tri
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
